# Radar
Please read the `README.md` file before running this tutorial.

This tutorial shows some basic radar data accessing and helper functions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from pyboreas import BoreasDataset
from pyboreas.utils.utils import get_inverse_tf

root = '/path/to/data/boreas/'
split = None
# AWS: Note: Free Tier SageMaker instances don't have enough storage (25 GB) for 1 sequence (100 GB)
# root = '/home/ec2-user/SageMaker/boreas/'
# split = [['boreas-2021-09-02-11-42', 163059759e6, 163059760e6-1]]

# With verbose=True, the following will print information about each sequence
bd = BoreasDataset(root, split=split, verbose=True)
# Grab the first sequence
seq = bd.sequences[0]

In [ ]:
# Radar is calibrated to other sensors through the Velodyne lidar
T_radar_lidar = seq.calib.T_radar_lidar
print('T_radar_lidar:\n', T_radar_lidar)
T_radar_applanix = T_radar_lidar @ get_inverse_tf(seq.calib.T_applanix_lidar)
print('T_radar_applanix:\n', T_radar_applanix)

In [ ]:
# The raw radar data is contained within a polar image.
# Note that the first 2.5 meters of the radar image are zeroed out by default
# to remove noise from the vehicle itself.

rad = seq.get_radar(0)
plt.figure(figsize=(20, 100))
polar_img = plt.imshow(rad.polar, cmap="gray")
plt.title("Polar Radar Intensity image")
plt.xlabel("Range Bins")
plt.ylabel("Azimuth Bins")
plt.show()

In [ ]:
# A more intuitive visualization is achieved through converting the polar image to a Cartesian image
# The radar visualizer plots this Cartesian image.
# Take a look inside the visualizer to see how the radar data gets converted to a Cartesian image
# There are also other options, including saving the cartesian image:
# rad.visualize(figsize=(5, 5), save='/path/to/saved/image/radar.png')

rad = seq.get_radar(0)
cart_img = rad.visualize(figsize=(3, 3))

In [ ]:
# Note that each sensor frame has a timestamp, pose (4x4 homogeneous transform), and velocity information.
rad = seq.get_lidar(0)
print('Radar:')
print('timestamp: {}'.format(rad.timestamp))
print('pose (T_enu_lidar):')
print(rad.pose)
print('velocity (wrt ENU):')
print(rad.velocity)
print('body rate (wrt sensor):')
print(rad.body_rate)

# Note that lidar and camera frames are collected at 10Hz, but radar frames collected at 4 Hz.

In [ ]:
# Each radar azimuth contains its own timestamp
# The radar frame timestamp corresponds to the middle of the frame
print('Radar at frame timestamp {} has {} azimuths.'.format(rad.timestamp_micro, len(rad.azimuths)))
print('Timestamps of the middle 5 azimuths:')
print(rad.timestamps[len(rad.azimuths) // 2 - 3: len(rad.azimuths) // 2 + 2])

In [ ]:
# Example: using an iterator
rad_iter = bd.sequences[0].get_radar_iter()
rad0 = next(rad_iter)  # First camera frame
rad1 = next(rad_iter)  # Second camera frame
print(rad0.timestamp)
print(rad1.timestamp)